# AI Talking Photo — Google Colab

單張人物照片 + 繁體中文講稿 → Edge TTS → FFmpeg → Wav2Lip → MP4。

**使用前請先在 Colab 選擇 GPU 執行階段，然後「全部執行」。**
Notebook 會建立獨立 Python 3.11 環境，不依賴 Colab 目前預裝的 Python 版本。

> 請只使用你有權使用的人像與內容。官方 Wav2Lip 公開模型限制個人、研究／學術、非商業用途。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

print("Colab kernel Python:", sys.version.split()[0])
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("未偵測到 NVIDIA GPU。請在「執行階段」選擇 GPU 後重新全部執行。")
subprocess.run(["nvidia-smi"], check=True)


## 1. 取得專案

每次從 GitHub `main` 取得最新版；若同一個 Colab 工作階段已存在目錄，就先更新。

In [ ]:
REPO_URL = "https://github.com/similaitw/ai-talking-photo.git"
REPO = Path("/content/ai-talking-photo")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Project:", REPO)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


## 2. 建立固定 Python 3.11 推論環境

Colab 的系統 Python 會更新，因此以 `uv` 建立專案自己的 Python 3.11 環境。
PyTorch 使用已在本專案真實驗證過的 2.5.1 + CUDA 11.8 組合，其餘套件依 lock file 安裝。

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv 安裝失敗，無法建立 Python 3.11 環境。")

VENV = REPO / ".colab-venv"
subprocess.run([UV, "python", "install", "3.11"], check=True)
if not VENV.exists():
    subprocess.run([UV, "venv", "--python", "3.11", "--seed", str(VENV)], check=True)

PYTHON = str(VENV / "bin" / "python")
subprocess.run(
    [PYTHON, "-m", "pip", "install", "torch==2.5.1",
     "--index-url", "https://download.pytorch.org/whl/cu118"],
    check=True,
)
subprocess.run(
    [PYTHON, "-m", "pip", "install", "-r", str(REPO / "requirements-wav2lip-lock.txt")],
    check=True,
)
subprocess.run([PYTHON, "-c", "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.is_available())"], check=True)


## 3. 準備官方 Wav2Lip 與模型

第三方程式與模型只放在 Colab 工作階段，不寫回 GitHub。
Wav2Lip 固定到本專案已驗證的官方 commit，並使用官方指定的 GAN 與 S3FD 下載來源。

In [ ]:
WAV2LIP = REPO / "vendor" / "Wav2Lip"
WAV2LIP_REVISION = "bac9a81e63ecc153202353372e5724b83d9e6322"

WAV2LIP.parent.mkdir(parents=True, exist_ok=True)
if not WAV2LIP.exists():
    subprocess.run(["git", "clone", "https://github.com/Rudrabha/Wav2Lip.git", str(WAV2LIP)], check=True)
subprocess.run(["git", "-C", str(WAV2LIP), "fetch", "--depth", "1", "origin", WAV2LIP_REVISION], check=True)
subprocess.run(["git", "-C", str(WAV2LIP), "checkout", "--force", WAV2LIP_REVISION], check=True)

models = REPO / "models"
models.mkdir(exist_ok=True)
checkpoint = models / "wav2lip_gan.pth"
official_copy = models / "wav2lip_gan.official.torchscript"
if not checkpoint.exists() and not official_copy.exists():
    subprocess.run(
        [PYTHON, "-m", "gdown", "15G3U08c8xsCkOqQxE38Z2XXDnPcOptNk", "-O", str(checkpoint)],
        check=True,
    )

detector = WAV2LIP / "face_detection" / "detection" / "sfd" / "s3fd.pth"
detector.parent.mkdir(parents=True, exist_ok=True)
if not detector.exists():
    subprocess.run(
        ["curl", "-L", "--fail",
         "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth",
         "-o", str(detector)],
        check=True,
    )

subprocess.run([PYTHON, str(REPO / "scripts" / "prepare_wav2lip.py")], cwd=REPO, check=True)
print("Wav2Lip 與模型準備完成。")


## 4. 環境診斷

Doctor 必須通過，才啟動介面。這一步不會產生影片。

In [ ]:
subprocess.run([PYTHON, str(REPO / "scripts" / "doctor.py")], cwd=REPO, check=True)


## 5. 啟動 Gradio

執行後請開啟輸出中的 `gradio.live` 公開網址。停止這個 cell 即可關閉服務。

In [ ]:
launch_code = """
import os
import sys
os.chdir(r'/content/ai-talking-photo')
sys.path.insert(0, os.getcwd())
from app import build_app
build_app().launch(
    share=True,
    server_name='0.0.0.0',
    show_error=True,
)
"""
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.run([PYTHON, "-u", "-c", launch_code], cwd=REPO, env=env, check=True)
